 <b><i> create_hosing_fields </i></b>


Created by Eduardo Alastrue de Asenjo on 2025-02-20

- Purpose: create time-varying hosing fields from existing files
- Method: follow Pontes & Menviel (2024), but expand to two different rates of increasing hosing 
- Comments: since th .config defines the cdo command applied for both flux and mask files, the years are also extended for the masks, even if they don't change over time


In [ ]:
import xarray as xr
import numpy as np
import dask
import glob
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import os; os.environ['PROJ_LIB'] = '/work/uo1075/m300817/phd/conda/share/proj'
#import regionmask
import xesmf as xe
import cdo # Import Cdo-py
cdo = cdo.Cdo(tempdir='/scratch/m/m300817/tmp/cdo-py') # change this to a directory in your scratch
import eccodes
import cfgrib
import zlib
from tqdm import tqdm
import pandas as pd

# Load files

In [ ]:
hos_g01 = xr.open_mfdataset("/work/mh0287/m211054/mpiesm/hosing/masks/HOSING_GRC_01SV_GR15.nc")
hos_g03 = xr.open_mfdataset("/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/HOSING_GRC_03SV_GR15.nc")
hos_gmask = xr.open_mfdataset("/work/mh0287/m211054/mpiesm/hosing/masks/HOSING_GRC_MASK_GR15.nc")

In [ ]:
fig=plt.figure(figsize=(14, 9))
ax1 = fig.add_subplot(211, projection=ccrs.Robinson(central_longitude=-60))
(hos_g01.flux.where((hos_g01.flux>0).compute(), drop=True)*1e6).plot(x='longitude', y='latitude', ax=ax1, transform=ccrs.PlateCarree(), vmin=0)
ax1.coastlines()

# historical

## flux

In [ ]:
# As in Pontes & Menviel (2024)
hos_g001 = hos_g01/10 # set the base to 0.01 Sv
years = pd.date_range(start='1850-01-01', end='2014-01-01', freq='YS')
hos_g001 = hos_g001.expand_dims(time=years)
ds = hos_g001
ds = ds.where((ds.time.dt.year < 1960) | (ds.time.dt.year > 1989), ds * 5) # ds.where(condition, other)
ds = ds.where((ds.time.dt.year < 1990) | (ds.time.dt.year > 1999), ds * 0)
ds = ds.where((ds.time.dt.year < 2000) | (ds.time.dt.year > 2004), ds * 5)
ds = ds.where((ds.time.dt.year < 2005) | (ds.time.dt.year > 2014), ds * 7.5)
ds.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_HIST_GR15.nc')

In [ ]:
p = ds.sel(time='1870-01-01')
fig=plt.figure(figsize=(14, 9))
ax1 = fig.add_subplot(211, projection=ccrs.Robinson(central_longitude=-60))
(p.flux.where((p.flux>0).compute(), drop=True)*1e6).plot(x='longitude', y='latitude', ax=ax1, transform=ccrs.PlateCarree(), vmin=0)
ax1.coastlines()

In [ ]:
p = ds.sel(time='1970-01-01')
fig=plt.figure(figsize=(14, 9))
ax1 = fig.add_subplot(211, projection=ccrs.Robinson(central_longitude=-60))
(p.flux.where((p.flux>0).compute(), drop=True)*1e6).plot(x='longitude', y='latitude', ax=ax1, transform=ccrs.PlateCarree(), vmin=0)
ax1.coastlines()

In [ ]:
p = ds.sel(time='2005-01-01')
fig=plt.figure(figsize=(14, 9))
ax1 = fig.add_subplot(211, projection=ccrs.Robinson(central_longitude=-60))
(p.flux.where((p.flux>0).compute(), drop=True)*1e6).plot(x='longitude', y='latitude', ax=ax1, transform=ccrs.PlateCarree(), vmin=0)
ax1.coastlines()

## mask 

In [ ]:
# As in Pontes & Menviel (2024)
years = pd.date_range(start='1850-01-01', end='2014-01-01', freq='YS')
hos_gmask_hist = hos_gmask.expand_dims(time=years)
hos_gmask_hist.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_MASKHIST_GR15.nc')

# scenarios

## flux

In [ ]:
# As in Pontes & Menviel (2024), 0.025 Sv increase per decade
hos_g001 = hos_g01/10 # set the base to 0.01 Sv
years = pd.date_range(start='2015-01-01', end='2100-01-01', freq='YS')
hos_g001 = hos_g001.expand_dims(time=years)
ds = hos_g001
ds = ds.where((ds.time.dt.year < 2015) | (ds.time.dt.year > 2019), ds * 7.5) # ds.where(condition, other)
ds = ds.where((ds.time.dt.year < 2020) | (ds.time.dt.year > 2029), ds * 10.0)
ds = ds.where((ds.time.dt.year < 2030) | (ds.time.dt.year > 2039), ds * 12.5)
ds = ds.where((ds.time.dt.year < 2040) | (ds.time.dt.year > 2049), ds * 15.0)
ds = ds.where((ds.time.dt.year < 2050) | (ds.time.dt.year > 2059), ds * 17.5)
ds = ds.where((ds.time.dt.year < 2060) | (ds.time.dt.year > 2069), ds * 20.0)
ds = ds.where((ds.time.dt.year < 2070) | (ds.time.dt.year > 2079), ds * 22.5)
ds = ds.where((ds.time.dt.year < 2080) | (ds.time.dt.year > 2089), ds * 25.0)
ds = ds.where((ds.time.dt.year < 2090) | (ds.time.dt.year > 2100), ds * 27.5)

ds.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_SCE_025SVDEC_GR15.nc')

In [ ]:
# As in Pontes & Menviel (2024), 0.010 Sv increase per decade
hos_g001 = hos_g01/10 # set the base to 0.01 Sv
years = pd.date_range(start='2015-01-01', end='2100-01-01', freq='YS')
hos_g001 = hos_g001.expand_dims(time=years)
ds = hos_g001
ds = ds.where((ds.time.dt.year < 2015) | (ds.time.dt.year > 2019), ds * 7.5) # ds.where(condition, other)
ds = ds.where((ds.time.dt.year < 2020) | (ds.time.dt.year > 2029), ds * 8.5)
ds = ds.where((ds.time.dt.year < 2030) | (ds.time.dt.year > 2039), ds * 9.5)
ds = ds.where((ds.time.dt.year < 2040) | (ds.time.dt.year > 2049), ds * 10.5)
ds = ds.where((ds.time.dt.year < 2050) | (ds.time.dt.year > 2059), ds * 11.5)
ds = ds.where((ds.time.dt.year < 2060) | (ds.time.dt.year > 2069), ds * 12.5)
ds = ds.where((ds.time.dt.year < 2070) | (ds.time.dt.year > 2079), ds * 13.5)
ds = ds.where((ds.time.dt.year < 2080) | (ds.time.dt.year > 2089), ds * 14.5)
ds = ds.where((ds.time.dt.year < 2090) | (ds.time.dt.year > 2100), ds * 15.5)

ds.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_SCE_010SVDEC_GR15.nc')

In [ ]:
# As in Pontes & Menviel (2024), 0.040 Sv increase per decade
hos_g001 = hos_g01/10 # set the base to 0.01 Sv
years = pd.date_range(start='2015-01-01', end='2100-01-01', freq='YS')
hos_g001 = hos_g001.expand_dims(time=years)
ds = hos_g001
ds = ds.where((ds.time.dt.year < 2015) | (ds.time.dt.year > 2019), ds * 7.5) # ds.where(condition, other)
ds = ds.where((ds.time.dt.year < 2020) | (ds.time.dt.year > 2029), ds * 11.5)
ds = ds.where((ds.time.dt.year < 2030) | (ds.time.dt.year > 2039), ds * 15.5)
ds = ds.where((ds.time.dt.year < 2040) | (ds.time.dt.year > 2049), ds * 19.5)
ds = ds.where((ds.time.dt.year < 2050) | (ds.time.dt.year > 2059), ds * 23.5)
ds = ds.where((ds.time.dt.year < 2060) | (ds.time.dt.year > 2069), ds * 27.5)
ds = ds.where((ds.time.dt.year < 2070) | (ds.time.dt.year > 2079), ds * 31.5)
ds = ds.where((ds.time.dt.year < 2080) | (ds.time.dt.year > 2089), ds * 35.5)
ds = ds.where((ds.time.dt.year < 2090) | (ds.time.dt.year > 2100), ds * 39.5)

ds.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_SCE_040SVDEC_GR15.nc')

In [ ]:
# As in Pontes & Menviel (2024), 0.040 Sv increase per decade
hos_g001 = hos_g01/10 # set the base to 0.01 Sv
years = pd.date_range(start='2015-01-01', end='2100-01-01', freq='YS')
hos_g001 = hos_g001.expand_dims(time=years)
ds = hos_g001
ds = ds.where((ds.time.dt.year < 2015) | (ds.time.dt.year > 2019), ds * 7.5) # ds.where(condition, other)
ds = ds.where((ds.time.dt.year < 2020) | (ds.time.dt.year > 2029), ds * 11.5)
ds = ds.where((ds.time.dt.year < 2030) | (ds.time.dt.year > 2039), ds * 15.5)
ds = ds.where((ds.time.dt.year < 2040) | (ds.time.dt.year > 2049), ds * 19.5)
ds = ds.where((ds.time.dt.year < 2050) | (ds.time.dt.year > 2059), ds * 23.5)
ds = ds.where((ds.time.dt.year < 2060) | (ds.time.dt.year > 2069), ds * 27.5)
ds = ds.where((ds.time.dt.year < 2070) | (ds.time.dt.year > 2079), ds * 31.5)
ds = ds.where((ds.time.dt.year < 2080) | (ds.time.dt.year > 2089), ds * 35.5)
ds = ds.where((ds.time.dt.year < 2090) | (ds.time.dt.year > 2100), ds * 39.5)

ds.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_SCE_040SVDEC_GR15.nc')

In [ ]:
p = ds.sel(time='2090-01-01')
fig=plt.figure(figsize=(14, 9))
ax1 = fig.add_subplot(211, projection=ccrs.Robinson(central_longitude=-60))
(p.flux.where((p.flux>0).compute(), drop=True)*1e6).plot(x='longitude', y='latitude', ax=ax1, transform=ccrs.PlateCarree(), vmin=0)
ax1.coastlines()

## mask

In [ ]:
# As in Pontes & Menviel (2024)
years = pd.date_range(start='2015-01-01', end='2100-01-01', freq='YS')
hos_gmask_sce = hos_gmask.expand_dims(time=years)
hos_gmask_sce.to_netcdf('/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/time_dep_hosing_fields/HOSING_GRC_MASKSCE_GR15.nc')

In [ ]:
fig=plt.figure(figsize=(14, 9))
ax1 = fig.add_subplot(211, projection=ccrs.Robinson(central_longitude=-60))
(hos_gmask_hist.isel(time=5).flux).plot(x='longitude', y='latitude', ax=ax1, transform=ccrs.PlateCarree(), vmin=0)
ax1.coastlines()